In [ ]:
# Copyright 2026 - Osmar Yupanqui & Marvin Quispe
# Conservación Amazónica - ACCA
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Selective Logging Detection with Deep Learning and Very-High Resolution Imagery

#### Authors: Osmar Yupanqui and Marvin Quispe

<br>

## Part 1: Data processing and exporting

### Due to the growing volume of available satellite data and the need for researchers and organizations to process it efficiently, cloud platforms such as [Google Earth Engine (GEE)](https://earthengine.google.com/). have become widely used tools in geospatial analysis. GEE allows users to access and process large collections of satellite data without needing to download them locally, which significantly reduces computational requirements; for this reason, it is currently one of the most used platforms for geospatial analysis. Additionally, it offers a free plan, as well as commercial options for users with greater processing and storage needs.

### This notebook presents a simple workflow for processing satellite data on the Google Earth Engine (GEE) platform and exporting it to an optimized format (TFRecord) to Google Drive. It should be noted that all notebooks in this application example have been developed to run on Google Colab (free version), using CPUs for reference data export and inference (Notebooks n.º1 and n.º3), and GPUs for model training (Notebook n.º2). Therefore, we recommend using this platform for execution. If you wish to run it locally, we suggest using Visual Studio Code, after installing the [Google Colab VS Code Extension](https://marketplace.visualstudio.com/items?itemName=Google.colab).

### All the files needed to implement this application example are stored in a [Google Drive folder](https://drive.google.com/drive/u/4/folders/1v30R0-2j5WkgekRxYUw4PHS04c2N9YJR) named `DL_Book`. We recommend downloading and reviewing its contents to better understand the data structure or, if you prefer, proceeding directly to model training (Notebook n.º2). However, it is not necessary to download it to run this notebook (Notebook n.º1), as it includes the process of exporting the reference data from scratch directly to Google Drive. The complete export of the reference data takes approximately 1 hour and 20 minutes and generates a set of files with a total size of approximately 17 MB.


 To avoid future complications and as a good practice, create the main working folder named `DL_Book` before running the main code. This folder should be located in the main storage path; in our case, on Google Drive.

In [ ]:
# We are going to connect Google Drive with Google Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Next, we create the main working folder
import os
os.makedirs('/content/drive/MyDrive/DL_Book', exist_ok = True)

Additionally, print the available computational resources in Google Colab to understand the characteristics of the environment in which the notebook will run.


In [ ]:
print("─── Google Colab resources report ───")
!echo -n "CPU Model: " && lscpu | grep "Model name" | cut -d':' -f2 | xargs
!echo -n "Total RAM: " && free -h | awk '/Mem:/ {print $2}'
!echo -n "Available Disk: " && df -h / | awk 'NR==2 {print $4 " of " $2}'
!echo -n "GPU: " && (nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo "Not available")

─── Google Colab resources report ───
CPU Model: Intel(R) Xeon(R) CPU @ 2.20GHz
Total RAM: 12Gi
Available Disk: 87G of 108G
GPU: Not available


## 1. Understanding the selective logging activity

Forest conservation faces a threat that is difficult to detect but highly devastating: **selective logging**. Unlike deforestation, which can be identified relatively easily from space using freely available satellite imagery, selective logging involves the removal of only **commercially valuable trees** from the forest cover, leaving a deceptive appearance of normality in the forest. For this reason, detecting this activity poses a considerable technological and logistical challenge.

First, the areas of forest loss caused by this activity are **typically not detected using medium-resolution remote sensing imagery** such as Sentinel-2 and Landsat. This is because tree removal in the forest leaves a small area of ​​impact. Furthermore, the impact is difficult to observe because it occurs below the forest canopy.

Second, there is only a **short window after harvesting to assess this activity** (2-3 months). This is because tropical forests, especially those in the Amazon rainforest, exhibit high rates of natural regeneration during vegetation succession processes. As a result, signs of disturbance can disappear quickly. Furthermore, the frequency of satellite image acquisition is not always sufficient to detect the activity in time, before it spreads to more remote areas.

Third, the **heavy cloud cover in the Amazon region**. Cloud cover limits the observation of land cover using optical sensors. In addition, in recent years there has been an increase in cloud cover in some areas of the Amazon; however, the causes of this phenomenon have not yet been thoroughly analyzed.

For this reason, researchers and organizations in the region, such as Conservación Amazónica - ACCA, use commercial satellite imagery from SkySat (0.5-meter spatial resolution), operated by Planet Labs, to monitor selective logging. These images can be acquired on demand and are updated daily, making them a particularly well-suited tool for monitoring remote areas.

Below is an image comparing, in the same location, logging activity with a SkySat image (0.5 m) and a drone image (3 cm). Logs are visible, and a blue canister can be seen for refilling fuel for the chainsaws used for logging.


![Logging_A](https://drive.google.com/uc?export=view&id=1LAV6IF03RXBrJOU9yfV5ynh5G3qa2MdB)
*Comparison between a SkySat image (0.5 m) and a drone image (3 cm)*

Selective logging is often associated to **infrastructures** such as logging roads, logging yards, camps, among others. Identifying these critical infrastructures makes it easier to detect logging activities. The presence of these infrastructures occurs in areas where logging is common, such as timber forest concessions and forest plantations. However, in remote areas where the activity is generally illegal, it is possible to observe a network of trails, usually beneath the forest canopy.

![Logging_B](https://drive.google.com/uc?export=view&id=1vEfrzinhfrF8MkVzxMnbh6mBTE2cXK1b)
*Diferent infrastructures associated with selective logging*

## 2. Setting up your Google Earth Engine account

The Google Earth Engine Python API requires a personal account. For enabling the API you need the following:
- A Google account linked to Google Earth Engine.
- A Google Cloud Project (A non-commercial project is suggested).


For linking your Google account with GEE, [log in](https://earthengine.google.com/) using your credentials with the **Get Started** button. This will take you to the [Google Cloud Console](https://console.cloud.google.com/), here you will have to register a project. We strongly suggest a non-commercial project, find out more about princing [here](https://cloud.google.com/earth-engine/pricing).

![Creating a Google Cloud Project A](https://drive.google.com/uc?export=view&id=14wl7mhCI-HTbVQrp__kxyghpb5tkvw0m)

Then, click on the create project button.
![Creating a Google Cloud Project B](https://drive.google.com/uc?export=view&id=1QC7C5zZIvGQH7M3l4JwoPpVjifB3mxj6)

After that, define the name of your project (remember this, it will be used later, also notice the numbers after your project, this will be the **true** ID of your project), and click on the create button.

![Creating a Google Cloud Project C](https://drive.google.com/uc?export=view&id=1GWweZsHomwsb43qrk7hDd87hB0TYcVW6)

Now you will see 2 different options for accessing Earth Engine: a commercial use (paid) and a noncommercial use. We will use non-commercial option for this workflow.
![Creating a Google Cloud Project D](https://drive.google.com/uc?export=view&id=1z-2QXIDP95aj2jLfTb7-nzvMaUHmBVt8)

There will be 5 steps that you will have to complete:
- Select your organization type - we suggest selecting either non-profit or other
- Check non-commercial eligibility - it is important to select **Individual research or non-commercial use (not involved with an organization)**, after that provide info about how you will be using Earth Engine, this could be research about land use, or other motive.
- Choose your plan - this step is not needed since you are already eligible for non-commercial use.
- Describe your work - you could choose protection & conservation, you can use Earth Engine for forestry (this step does not matter much, since you are already eligible)
- Review summary - verify that everything is ok and click on the register button.

![Creating a Google Cloud Project E](https://drive.google.com/uc?export=view&id=1xpHaKo5H1kW7Je4IJBbY_q55HlZbMbRe)

You will see a screen that displays that your Google Earth Engine API is not enabled, click on Enable.

![Creating a Google Cloud Project F](https://drive.google.com/uc?export=view&id=12PntnVViS_f9f2a2Dnqou5yqaEhGI4n4)

Finally, verify that you have a confirmation screen that confirms your non-commercial registration.
![Creating a Google Cloud Project G](https://drive.google.com/uc?export=view&id=1uOC2HFQ1BxfkpNxbw61Z3NFWRpE9_XIr)

**Do not forget your project ID**, you can find it here (we will use this later).
![Creating a Google Cloud Project H](https://drive.google.com/uc?export=view&id=1vmgJdyoWW0Kuc-EGRCqEiyzQYzKNM7Lh)

It is also important to note that, effective April 27, 2026, Earth Engine has established non-commercial quota levels for its use. These quotas are assigned at the project level and are renewed monthly, measured in Earth Engine Compute Units (EECU). Therefore, it is necessary to select an appropriate quota level based on the project’s requirements. For more information on quota levels, please refer to the [official documentation](https://developers.google.com/earth-engine/guides/noncommercial_thresholding?hl=en_US).

![Select Quota Tier A](https://drive.google.com/uc?export=view&id=1_eXdTqowB14lx8IGs34TDj3c1LusVoBX)

For running this application example, the free “Community” quota level is sufficient, as it offers a limit of 150 EECU-hours. In this case, exporting the reference data to Google Drive consumes approximately 110 EECU-seconds, which is well within that limit.

![Select Quota Tier B](https://drive.google.com/uc?export=view&id=1QsOVr8q6DbGEzNgn1CyVYzo0vrp0iHlz)

![Selective Logging EECU table](https://drive.google.com/uc?export=view&id=1mWPOGbSsiQeBzSEt0rtuCKvVG51Hmscg)


## 3. Data exploration

First, we will import the official [Earth Engine library](https://developers.google.com/earth-engine/guides/python_install) in **Python**, for the data visualization and processing, using `import`

In [ ]:
import ee

Because the Earth Engine API requires a personal account, we will proceed to authenticate our account and initialize Earth Engine.  
When we run the following cell, a link will appear, where we must provide authorizations. Note that we have to introduce our **project ID** in `ee.Initialize()`.

In [ ]:
ee.Authenticate()
# Replace 'my-project' with your project id
ee.Initialize(project = 'my-project')

Finally, we will import another important library called [Geemap](https://geemap.org/), which will allow us to visualize the data using [ipyleaflet](https://github.com/jupyter-widgets/ipyleaflet).  If we do not have **geemap**, we can download it using the `!pip install` command.

In [ ]:
import geemap

Using the Earth Engine API we can visualize the data using python and geemap. We will load a very high resolution (0.5 m) SkySat satellite image as well as the collected training data (polygons).

In [ ]:
img = ee.Image('projects/ACCA-SERVIR/Colaboraton/Skysat_Fatima') # This imports the SkySat image as an ee.Image()
polygons = ee.FeatureCollection('projects/ACCA-SERVIR/Colaboraton/Skysat_Fatima_pols') # This imports the data (polygons) that will be further used in the training, as an ee.FeatureCollection()

The **Geemap library** is very useful for visualizing our data.

In [ ]:
Map = geemap.Map(center = (-9.653, -74.198), zoom = 15) # This creates a map using ipyleaflet
Map.addLayer(img, {'bands': ['b3', 'b2', 'b1'], 'min': 2000, 'max': 8000}, 'SkySat') # This adds our image to the map, with custom visualization parameters
Map.addLayer(polygons, {'color': 'red'}, 'Training Polygons') # This adds the imported polygons to the map
Map

Map(center=[-9.653, -74.198], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDat…

We will also print the amount of polygons that we have.

In [ ]:
print(polygons.size().getInfo()) # The .size() method retrieves the amount of polygons in our feature collection, .getInfo() brings the response to the client synchronously

246


## 4. Data processing

### 4.1 Assign a numerical value to the class

This workflow is designed for a binary semantic segmentation, for this, we need an image label, that matches the projection and size of our input image. This image label will require two values:
- Value of 1: this is what we want to detect (logging).
- Value of 0: the background (forest, water, etc.).

Our database consists of polygons, which all have a string attribute called **tipo** (type in english), which is not numeric. We will create a function to create an attribute called **class** with the value of 1 in our feature collection. If this exercise were a multiclass segmentation, more types besides **tala** (logging) would be expected.

We will print the attributes of the first polygon. Note that it is advisable to have attributes that refer to the image used in the digitalization of the polygons.

*Notice the attributes at the end*.

In [ ]:
print(polygons.first().getInfo())

{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-74.20131834249987, -9.646241075424708], [-74.20131395355791, -9.646245561421832], [-74.20130938547537, -9.646245677105858], [-74.20130490696295, -9.646245714675706], [-74.20130042845042, -9.64624575224548], [-74.20129586036754, -9.646245867929274], [-74.20129138185479, -9.6462459054989], [-74.20128681377167, -9.646241494641858], [-74.20128224568846, -9.64623708378515], [-74.20128215611818, -9.64623263535913], [-74.20128663463116, -9.646228071249496], [-74.20128654506091, -9.646223622823896], [-74.20129102357377, -9.646223585253951], [-74.20130007016941, -9.646223431999763], [-74.20130454868195, -9.646223394429596], [-74.20130911676462, -9.646223278745268], [-74.20131816335939, -9.646227652030708], [-74.20131825292962, -9.646232100457091], [-74.20131834249987, -9.646241075424708]]]}, 'id': '00000000000000000000', 'properties': {'id': 24, 'imgID': 'Skysat_20201017_ssc2_150055', 'tipo': 'tala', 'type': 'Polygon'}}


This creates the function to create an attribute called **class** with the value of 1. Since all of our polygons are logging, a filter won't be needed.

In [ ]:
def add1(f):
    return f.set('class', 1.0) # The .set() method creates an attribute in each feature

We will print the first feature in our collection, notice the presence of the new attribute created.

In [ ]:
polygons = polygons.map(add1) # With the .map() method we apply a function to each feature inside of our feature collection
print(polygons.first().getInfo())

{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-74.20131834249987, -9.646241075424708], [-74.20131395355791, -9.646245561421832], [-74.20130938547537, -9.646245677105858], [-74.20130490696295, -9.646245714675706], [-74.20130042845042, -9.64624575224548], [-74.20129586036754, -9.646245867929274], [-74.20129138185479, -9.6462459054989], [-74.20128681377167, -9.646241494641858], [-74.20128224568846, -9.64623708378515], [-74.20128215611818, -9.64623263535913], [-74.20128663463116, -9.646228071249496], [-74.20128654506091, -9.646223622823896], [-74.20129102357377, -9.646223585253951], [-74.20130007016941, -9.646223431999763], [-74.20130454868195, -9.646223394429596], [-74.20130911676462, -9.646223278745268], [-74.20131816335939, -9.646227652030708], [-74.20131825292962, -9.646232100457091], [-74.20131834249987, -9.646241075424708]]]}, 'id': '00000000000000000000', 'properties': {'class': 1, 'id': 24, 'imgID': 'Skysat_20201017_ssc2_150055', 'tipo': 'tala', 'type': 'Pol

### 4.2 Create the "class" band of the image

As we mentioned before, we need an image label. For this, we will create a band, using our processed polygons. This image label will have two pixel values, 1 for logging and 0 for everything else (background).

To do this, we will create a function that [transforms](https://developers.google.com/earth-engine/apidocs/ee-featurecollection-reducetoimage) our feature collection (polygons) to a binary image.

In [ ]:
def createLabel (feature):
    featImg = feature.reduceToImage(['class'], ee.Reducer.mean()) # The .reduceToImage() method transforms our polygons to an image, with a mean reducer
    return featImg.toByte().rename('class') # We will return the image with a byte representation, this is memory efficient

Now we apply the createLabel() function and add the created band to our SkySat image.

In [ ]:
img = img.addBands(createLabel(polygons)) # The .addBands() method adds an image (or band) to another image

Again, we print the bands in the image. Note the existence of the band **class**, which will become the label, in this case logging.

In [ ]:
print(img.bandNames().getInfo())

['b1', 'b2', 'b3', 'b4', 'class']


Let's visualize our newly created label! You will see that it is very similar to the previou ipyleaflet map. But now our label is an image, instead of a feature collection (polygons).

In [ ]:
Map = geemap.Map(center = (-9.653, -74.198), zoom = 15) # This creates a map using ipyleaflet
Map.addLayer(img, {'bands': ['b3', 'b2', 'b1'], 'min': 2000, 'max': 8000}, 'SkySat') # This adds our image to the map, with custom visualization parameters
Map.addLayer(img.select('class'), {'min': 0, 'max': 1, 'palette': ['red']}, 'Image label') # This is our target, the image label, selected from the image
Map

Map(center=[-9.653, -74.198], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDat…

### 4.3 Image transformation

There are 3 ways of exporting our data:
- Exporting the whole image, then with a GIS Software or with programming, split the whole image into patches of equal size.
- Using the .getPixels() [method](https://developers.google.com/earth-engine/apidocs/ee-data-getpixels).
- Transforming our image to a neighborhood image, then exporting individual points (we will use this).

For the third method, we will transform our SkySat image (including the class band). In this neighborhood image, each pixel is a matrix representation of its neighboring pixels. This allows that when exporting a point, the neighborhood its contained as attributes (tensors).

For this, a kernel or matrix is needed. To build a kernel we will use a repetition of lists, with the value of 1 (this value is the weight that each component of the matrix will have).

Note that the size of the kernel will be the size of each exported patch. It is very important that the patch size/kernel contains the object that we want to detect. For this exercise we will export image patches with a size of 128 by 128.

In [ ]:
row = ee.List.repeat(1, 128) # This creates a list of 128 numbers with the value of 1
rows = ee.List.repeat(row, 128) # This repeats 128 times the previously creataed list
kernel = ee.Kernel.fixed(128, 128, rows) # This creates our kernel, with a size of 128.

We can print our kernel. Note that it has a size of 128 pixels wide, and 128 pixels height.

In [ ]:
print(kernel.getInfo())

{'type': 'Kernel.fixed', 'width': 128, 'height': 128, 'weights': '\n  [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]\n  [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 

Using the kernel, we can perform the image transformation, with the `.neightborhoodToArray()` method.

In [ ]:
img_final = img.neighborhoodToArray(kernel, 0) # The value of 0 adds the background value to the masked pixels

Finally, we will select the predictor variables, which will be part of the export and training.

In [ ]:
featureNames = ['b1', 'b2', 'b3', 'b4', 'class']
img_final = img_final.select(featureNames)

To verify, we will print the bands.

In [ ]:
print(img_final.bandNames().getInfo())

['b1', 'b2', 'b3', 'b4', 'class']


## 5. Data export

In machine learning and deep learning, it is common practice to divide the benchmark dataset into training, validation, and test subsets. The **training set** is used to adjust the model’s parameters, that is, to enable the algorithm to learn the patterns present in the data. The **validation set** is used to adjust the hyperparameters and monitor the model’s performance during training, allowing for the detection of issues such as overfitting. Finally, the **test set** is reserved for conducting a final, independent evaluation of the model, providing an objective estimate of its ability to generalize to unseen data.

The purpose of splitting the benchmark data is to ensure adequate model performance and to objectively evaluate that performance during the training process; therefore, this split is essential and is guided by various strategies and methodological criteria.

For example, a widely used ratio in machine learning, due to its alignment with the Pareto principle, is the 80:20 split between training and testing. From this basic configuration, multiple variations have emerged, most of which now incorporate a validation set. These ratios depend, to a large extent, on factors such as the amount of available data, the heterogeneity of the information, and the complexity of the problem being modeled. Consequently, it is possible to adjust the proportion of data allocated to training by reducing or expanding the validation and test sets according to the specific needs of the analysis, but, generally, without straying too far from the most common ratios, such as 80:20 (training/testing) or its variant, 70:20:10 (training/validation/testing). An inappropriate data splitting strategy can compromise the proper evaluation of the model and lead to unreliable performance estimates.

In this workflow, the dataset (polygons) will be divided into training (70%), validation (20%), and test (10%) subsets. Similar proportions have been used in other studies; for example, [Slagter et al. (2024)](https://www.sciencedirect.com/science/article/pii/S0034425724004061?via%3Dihub) used a 75/15/10 (training/validation/testing) split in the monitoring of forest roads in the Republic of the Congo.

First, we will assign a random value to each polygon using the `.randomColumn()` method. Then, we will sort the dataset based on this random value and split the polygons into training, validation, and test subsets.

In [ ]:
# Create random column
pols_final = polygons.randomColumn('random', seed=42)  # This creates a column in our feature collection, each feature will have a random value between 0 and 1.

# Sort polygons by the random value
pols_sorted = pols_final.sort('random')  # Sorting ensures a reproducible and deterministic split

print(pols_sorted.first().getInfo())

{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-74.19996457288495, -9.650228302766866], [-74.19996009433991, -9.65023286698335], [-74.19996018391083, -9.650237315607226], [-74.19995570536568, -9.6502418798239], [-74.19995131639132, -9.650246365924962], [-74.19994674827505, -9.650246481517282], [-74.19993788075492, -9.65025108739904], [-74.19993331263832, -9.650251120686692], [-74.19992435554661, -9.650251277943612], [-74.1999198770006, -9.650255842159583], [-74.19991539845448, -9.650255879635601], [-74.19991091990825, -9.650260443851433], [-74.1999064413619, -9.650260481327221], [-74.19990196281546, -9.650265045542925], [-74.19989748426889, -9.650265083018484], [-74.19989757383983, -9.650269531643028], [-74.19989309529318, -9.650274095858764], [-74.19988861674639, -9.650274133334023], [-74.19988852717546, -9.650269684709388], [-74.19988852717546, -9.650265240273754], [-74.19988843760453, -9.65026079164961], [-74.19988834803358, -9.650256260721363], [-74.199888258

We can see that the first polygon now has an additional attribute called **random**, which is used to reproducibly sort the dataset. Based on this ordered random value, we then split the dataset into training, validation, and test subsets using exact proportions.

In [ ]:
# Get total number of polygons
n = pols_sorted.size()

# Compute exact split sizes
n_train = n.multiply(0.7).int()  # 70% for training
n_val = n.multiply(0.2).int()    # 20% for validation
# The remaining polygons will be used for testing

# Convert FeatureCollection to list for slicing
pols_list = pols_sorted.toList(n)

# Create exact partitions
training_pols = ee.FeatureCollection(pols_list.slice(0, n_train))
validation_pols = ee.FeatureCollection(pols_list.slice(n_train, n_train.add(n_val)))
testing_pols = ee.FeatureCollection(pols_list.slice(n_train.add(n_val), n))

We can verify the number of training, validation and test polygons.

In [ ]:
print(training_pols.size().getInfo())
print(validation_pols.size().getInfo())
print(testing_pols.size().getInfo())

172
49
25


We will create lists in which the polygons will be included.

In [ ]:
trainingList = training_pols.toList(training_pols.size())
validationList = validation_pols.toList(validation_pols.size())
testingList = testing_pols.toList(testing_pols.size())

Finally, the data is exported to **Google Drive** (it can also be exported to Google Cloud Platform or an asset). A loop will be used to export a random pixel of our processed image, that intersects with the polygons, into a TFRecord file (this is the sampling). Remember that each pixel of the image is a representation of its neighborhood (128 by 128), so we are exporting patches as a TFRecord. We will use the `for` statement for this.

*Notice that we are exporting our data into a Google Drive folder called DL_Book, feel free to change it but remember the path*.

In [ ]:
for i in range(training_pols.size().getInfo()):
    sample = img_final.sample( # The .sample() method samples a feature inside a region
        region = ee.Feature(trainingList.get(i)).geometry(), # This defines the region, in this case, inside each polygon
        scale = 0.5, # The scale, SkySat have a 0.5 m of spatial resolution
        numPixels = 1, # We will only be exporting 1 pixel/1 patch of 128 by 128 per polygon, you can adjust this value to increase the number of patches per polygon
        seed = i,
        tileScale = 8
    )
    trainingTask = ee.batch.Export.table.toDrive(
        collection = sample, # We will export the sample (an individual point inside each of our polygons)
        description = 'train_' + str(i),
        folder = 'DL_Book', # The Google Drive folder that will contain the patch (Feel free to change it)
        fileNamePrefix = 'train_' + str(i), # The name of the file
        fileFormat = 'TFRecord', # The format, currently only TFRecord and GeoTIFF are supported
        selectors = featureNames # The bands that will be exported
    )
    trainingTask.start()
print('Training Data Exported')

for i in range(validation_pols.size().getInfo()):
    sample = img_final.sample(
        region = ee.Feature(validationList.get(i)).geometry(),
        scale = 0.5,
        numPixels = 1,
        seed = i,
        tileScale = 8
    )
    validationTask = ee.batch.Export.table.toDrive(
        collection = sample,
        description = 'validation_' + str(i),
        folder = 'DL_Book',
        fileNamePrefix = 'validation_' + str(i),
        fileFormat = 'TFRecord',
        selectors = featureNames
    )
    validationTask.start()
print('Validation Data Exported')

for i in range(testing_pols.size().getInfo()):
    sample = img_final.sample(
        region = ee.Feature(testingList.get(i)).geometry(),
        scale = 0.5,
        numPixels = 1,
        seed = i,
        tileScale = 8
    )
    testingTask = ee.batch.Export.table.toDrive(
        collection = sample,
        description = 'testing_' + str(i),
        folder = 'DL_Book',
        fileNamePrefix = 'testing_' + str(i),
        fileFormat = 'TFRecord',
        selectors = featureNames
    )
    testingTask.start()
print('Test Data Exported')

Training Data Exported
Validation Data Exported
Test Data Exported


You can check the status of your exports in the Google Earth Engine Code Editor, specifically in the Tasks tab. There, you can view the status of each of your tasks. It’s important to monitor them to identify potential export errors or check their progress.

![GEE Tasks](https://drive.google.com/uc?export=view&id=1-mIqdCx6ffYntD0Q1wT-vZM7-bNT19Gx)

Additionally, there are options for monitoring task status in a more advanced way. Currently, it has also become important to evaluate the quota consumption (EECU) associated with exports. For this type of more detailed analysis, we recommend using the Google Cloud Platform, accessing the Tasks section and supplementing it with "Monitoring Tool" to explore metrics and platform resource usage.


![GEE Tasks GCP](https://drive.google.com/uc?export=view&id=1jBcTgQ3DIPGFY4UOIOUAOYS4yh04uTvE)

The workflow summarizes the steps followed in this notebook.


![Workflow_A](https://drive.google.com/uc?export=view&id=1gKIun2ohUBV0oFInXjqaFZcZkACwUmS3)

## References

- Gorelick, N., Hancher, M., Dixon, M., Ilyushchenko, S., Thau, D., & Moore, R. (2017). Google Earth Engine: Planetary-scale geospatial analysis for everyone. Remote Sensing of Environment, 202, 18–27. https://doi.org/10.1016/j.rse.2017.06.031
- Planet Labs Inc. (2026). SkySat Imagery Products. https://developers.planet.com/docs/data/skysat/
- Wu, Q. (2020). geemap: A Python package for interactive mapping with Google Earth Engine. Journal of Open Source Software, 5(51), 2305. https://doi.org/10.21105/joss.02305
- Yupanqui Carrasco, O., & Quispe Sedano, M. J. (2026). Selective Logging Detection with Deep Learning and Very-High Resolution Imagery_TF_records [Data set]. Zenodo. https://doi.org/10.5281/zenodo.19614389